# 03 - Interogari Spark SQL

Scop: rulam 12 interogari SQL relevante pe datasetul curatat. Sunt incluse agregari, filtrari si un join simplu.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

spark = (
    SparkSession.builder
    .appName("FlightsProject")
    .master("spark://master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)


## Incarcare dataset curatat

Notebook-ul citeste CSV-ul curatat salvat de `02_eda_cleanup.ipynb`. Daca nu exista, trebuie rulat notebook-ul 02.


In [ ]:
CLEAN_CSV_PATH = "hdfs://master:9000/flights/processed/flights_clean_csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CLEAN_CSV_PATH)
    .repartition(4)
)

df.cache()
print("Randuri:", df.count())
df.createOrReplaceTempView("flights")
df.show(5, truncate=False)


## Tabela auxiliara pentru join

Construim o tabela mica `carriers` din dataset, cu cod si nume companie.


In [ ]:
carriers = df.select("carrier", F.col("name").alias("carrier_name")).dropDuplicates()
carriers.createOrReplaceTempView("carriers")
carriers.show(20, truncate=False)


## Q1 - Numar total de zboruri


In [ ]:
result = spark.sql("""
SELECT COUNT(*) AS total_flights FROM flights
""")
result.show(30, truncate=False)


## Q2 - Zboruri pe carrier


In [ ]:
result = spark.sql("""
SELECT carrier, name, COUNT(*) AS flights FROM flights GROUP BY carrier, name ORDER BY flights DESC
""")
result.show(30, truncate=False)


## Q3 - Intarziere medie la plecare si sosire pe carrier


In [ ]:
result = spark.sql("""
SELECT carrier, ROUND(AVG(dep_delay),2) AS avg_dep_delay, ROUND(AVG(arr_delay),2) AS avg_arr_delay FROM flights GROUP BY carrier ORDER BY avg_arr_delay DESC
""")
result.show(30, truncate=False)


## Q4 - Top 10 rute dupa numar de zboruri


In [ ]:
result = spark.sql("""
SELECT origin, dest, route, COUNT(*) AS flights FROM flights GROUP BY origin, dest, route ORDER BY flights DESC LIMIT 10
""")
result.show(30, truncate=False)


## Q5 - Rute cu cele mai mari intarzieri medii


In [ ]:
result = spark.sql("""
SELECT route, COUNT(*) AS flights, ROUND(AVG(arr_delay),2) AS avg_arr_delay FROM flights GROUP BY route HAVING flights >= 20 ORDER BY avg_arr_delay DESC LIMIT 10
""")
result.show(30, truncate=False)


## Q6 - Zboruri pe luna


In [ ]:
result = spark.sql("""
SELECT month, COUNT(*) AS flights, ROUND(AVG(arr_delay),2) AS avg_arr_delay FROM flights GROUP BY month ORDER BY month
""")
result.show(30, truncate=False)


## Q7 - Zboruri pe ora programata


In [ ]:
result = spark.sql("""
SELECT hour, COUNT(*) AS flights, ROUND(AVG(dep_delay),2) AS avg_dep_delay FROM flights GROUP BY hour ORDER BY hour
""")
result.show(30, truncate=False)


## Q8 - Procent zboruri intarziate la sosire pe carrier


In [ ]:
result = spark.sql("""
SELECT carrier, COUNT(*) AS flights, ROUND(100.0 * AVG(is_arrival_delayed), 2) AS pct_arrival_delayed FROM flights GROUP BY carrier ORDER BY pct_arrival_delayed DESC
""")
result.show(30, truncate=False)


## Q9 - Distanta medie pe destinatie


In [ ]:
result = spark.sql("""
SELECT dest, COUNT(*) AS flights, ROUND(AVG(distance),2) AS avg_distance FROM flights GROUP BY dest HAVING flights >= 20 ORDER BY avg_distance DESC LIMIT 15
""")
result.show(30, truncate=False)


## Q10 - Performanta pe aeroport de origine


In [ ]:
result = spark.sql("""
SELECT origin, COUNT(*) AS flights, ROUND(AVG(dep_delay),2) AS avg_dep_delay, ROUND(AVG(arr_delay),2) AS avg_arr_delay FROM flights GROUP BY origin ORDER BY flights DESC
""")
result.show(30, truncate=False)


## Q11 - Join intre tabela flights si tabela carriers


In [ ]:
result = spark.sql("""
SELECT f.carrier, c.carrier_name, COUNT(*) AS flights, ROUND(AVG(f.arr_delay),2) AS avg_arr_delay FROM flights f JOIN carriers c ON f.carrier = c.carrier GROUP BY f.carrier, c.carrier_name ORDER BY flights DESC
""")
result.show(30, truncate=False)


## Q12 - Categorii de intarziere


In [ ]:
result = spark.sql("""
SELECT CASE WHEN arr_delay <= 0 THEN 'early_or_on_time' WHEN arr_delay <= 15 THEN 'small_delay' WHEN arr_delay <= 60 THEN 'medium_delay' ELSE 'large_delay' END AS delay_category, COUNT(*) AS flights FROM flights GROUP BY CASE WHEN arr_delay <= 0 THEN 'early_or_on_time' WHEN arr_delay <= 15 THEN 'small_delay' WHEN arr_delay <= 60 THEN 'medium_delay' ELSE 'large_delay' END ORDER BY flights DESC
""")
result.show(30, truncate=False)


## Concluzie SQL

Interogarile arata diferente intre companii, rute, luni, ore si aeroporturi. Pentru proiect, acestea demonstreaza folosirea Spark SQL cu agregari, filtrari si join.
